# What shape do one-hot and frequency encoding actually have on this data?

*Setup cells below are carried from the shared analysis so this notebook runs on its own.*


In [1]:
import os

import polars as pl

from fraud_detection.core.feature_contract.admission import load_admission_rules
from fraud_detection.feature_engineering.derivations import load_frequency_maps

# Same modules the pipeline runs. That is the point of the layering rule in
# docs/code-structure.md: this notebook cannot measure a different implementation from the
# one that gets promoted, because there is only one.
PROJECT = os.environ["GCP_PROJECT_ID"]
SEEDS = [42, 7, 1337, 2024, 91]

## 1. What the encodings actually look like

Before measuring anything, the shape of the two families — because the shape is the whole
argument for why one of them is expected to fail.

In [2]:
rules = load_admission_rules()
declared = pl.DataFrame(
    [{"name": d.name, "tool": d.tool, "input": d.inputs[0]} for d in rules.derivations]
)
print(declared.group_by("tool").len().sort("len", descending=True))

maps = load_frequency_maps()
summary = pl.DataFrame(
    [
        {"column": c, "levels_in_map": len(t), "one_hot_columns_this_would_need": len(t)}
        for c, t in maps.items()
    ]
).sort("levels_in_map", descending=True)
summary

shape: (3, 2)
┌─────────────────────────┬─────┐
│ tool                    ┆ len │
│ ---                     ┆ --- │
│ str                     ┆ u32 │
╞═════════════════════════╪═════╡
│ one_hot                 ┆ 18  │
│ days_since_to_start_day ┆ 7   │
│ frequency_encode        ┆ 5   │
└─────────────────────────┴─────┘


column,levels_in_map,one_hot_columns_this_would_need
str,i64,i64
"""DeviceInfo""",1176,1176
"""addr1""",202,202
"""id_31""",99,99
"""R_emaildomain""",60,60
"""P_emaildomain""",59,59
